### Modeling Earth's Climate (Pfalzgraff, Eklof & Neshyba, 2026)
# Diagnostic Functions

## Three types of modeled variable
Here we are laying the groundwork for a revision of Cambio that provides little more organizational structure to our model. To do this, it's useful to distinguish between three different types of variables in climate modeling: *parametric*, *prognostic*, and *diagnostic*.

1. *Parametric variables* are variables that define a given climate model, such as the anthropogenic emission schedule or the climate sensitivity parameter. 
1. *Prognostic variables* are variables that are used to predict the future evolution of the climate, including (crucially) their own values! In our case, prognostic variables are concentrations of carbon in the carbon reservoirs (atmosphere and oceans), and the fluxes between these reservoirs. 
1. *Diagnostic variables* are values that can be calculated from parametric and prognostic variables that we're interested in, but are not otherwise essential to the model. Examples include the global temperature and ocean pH -- important for the health of the planet, but not needed by the Euler algorithm to calculate concentrations and fluxes as the model progresses from one year to the next.

(We should add that it's possible that as we re-envision our model, we could convert some variables from one type to another. For example, suppose we wanted to recognize a perfectly plausible feedback loop in which humans actually modified their future carbon emissions in response to the warming caused by past carbon emissions. That would promote global temperature from a diagnostic variable to a prognostic one. But we'll tackle that kind of thing in Cambio3.0.)

## Using libraries to stay organized
A library is a collection of code or functions that is intended to be used again and again.  As we do more with Cambio, it's helpful to organize things we do often into a *library*.  Here we call the library "MECLib", or "CL" for short.  In this notebook we'll use it to read in our scenario.  In the future we'll do more with this library.  

## Using dictionaries to organize data
Keeping track of all the variables and metadata involved in this can be very messy business. We've already seen one way to reduce that messiness in our emissions schedules through the use of structured variables defined by *Python Dictionaries*. We'll continue that idea here, by collecting *parametric* variables into a Python dictionary called *ClimateParams*.

## Using Python functions to compute diagnostic variables
Going hand in hand with the structural revisions described above, it will be useful to organize how we compute diagnostic variables. We'll do this here by means of Python functions, one function for each diagnostic variable. As you'll see, some of these will be pretty trivial, like adding a constant to a temperature anomaly to compute an actual temperature. And in other cases, the algorithms will get more complex. Here, we'll be using Python functions to take care of that -- along with validation tests (also called "unit tests") that make sure their algorithms are working properly.

## Uploading your emissions file
You'll need to upload your favorite emissions scenario into this space. 

## Learning goals
1. From inspecting the Python code, I can say whether a variable is prognostic, diagnostic, or parameteric.
1. I'm getting really good at creating Python functions.
1. I can extract information from Python dictionaries, and tailor them too.

In [ ]:
%pip install -q ipympl

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import pandas as pd              # MECLib uses these three packages, so we load them first
import matplotlib.pyplot as plt

# MECLib.py lives one level up, in the top-level folder of our Jupyter. This tells Python to look there.
import sys
sys.path.append('..')
import MECLib as cl

<!-- BEGIN QUESTION -->

### Load your scenario
To complete this exercise, you'll need an emissions scenario.  You can upload one, or use one from a previous week or ScenarioLibrary.  

Once you've done that, use the library function cl.LoadMyScenario to load the scenario into Python as a dictionary named epsdictionary_fromfile. Something like this should work:

    
    filename = '../ScenarioLibrary/Peaks_in_2040.pkl'       # or change this to whatever scenario you like
    time, eps, epsdictionary_fromfile = cl.LoadMyScenario(filename, verbose=True)

This is a lot simpler than what we were doing before - all the code we used to load scenarios before now lives in the library. 

_Points:_ 2

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Creating a ClimateParams dictionary
In the cell below, use the CreateClimateParams function to create a dictionary. Assuming you named your emissions dictionary "epsdictionary_fromfile," a command like the following should do it:

    ClimateParams = cl.CreateClimateParams(epsdictionary_fromfile)

Then, use display(ClimateParams) to make sure you did that correctly.

_Points:_ 2

In [ ]:
...

<!-- END QUESTION -->

### Tailoring your ClimateParams dictionary
It's easy to add items to a dictionary. Below is an example -- the item 'Description' is added.

In [ ]:
# Create a description 
ClimateParams['Description'] = "This scenario, authored by ..., specifies an albedo tipping point temperature of 2 degrees"

# Check that it got added correctly
display(ClimateParams)

<!-- BEGIN QUESTION -->

### Your turn
In the cell below, replace the 'Description' field with something that has your name in it. Then use the display command to verify it was added properly.

_Points:_ 1

In [ ]:
...

<!-- END QUESTION -->

## Diagnostic algorithms in functional form
In this part, you'll be writing Python functions that calculate values of diagnostic variables. For each algorithm, we have described the algorithm in mathematical terms. Your task is implement the algorithm as a Python function, as shown in the example below.

We'll also be running our functions through some benchmark tests to validate that they're working correctly, a process called "unit testing". The first unit test is done for you below, the second is up to you to implement.

<!-- BEGIN QUESTION -->

### Temperature in Fahrenheit from temperature in Celsius
Algorithm: $T(F) = T(C) \times 9 /5 + 32$, where $T(C)$ is in degrees Celsius.

Benchmark values:
- a temperature of $32 \ F$ if $T(C)=0 \ C$ (freezing temperature of water)
- a temperature of $212 \ F$ if $T(C)=100 \ C$ (boiling temperature of water)

_Points:_ 2

In [ ]:
def Diagnose_degreesF(T_C):
    """Returns an actual temperature in degrees F from an actual temperature in degrees C"""

    # Do the conversion to F
    T_F = T_C*9/5+32

    # Return the diagnosed temperature in F
    return(T_F)

# Benchmarking for freezing water (T = 0 degrees C)
print(Diagnose_degreesF(0))

# Benchmarking for boiling water (T = 100 degrees C)
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Temperature anomaly
Algorithm: 

$$
T_{anomaly} = CS\times(C_{atm}-C_{atm,pre-industrial}) + AS\times(\alpha-\alpha_{pre-industrial})
$$

where $CS$ is the climate sensitivity parameter (degrees warming per GtC increase in atmospheric $CO_2$), and $AS$ is the albedo sensitivity parameter (degrees warming per unit increase in albedo). Note that you'll have to extract the following parameters from ClimateParams (an example is provided in the cell below):

- 'climate sensitivity'
- 'preindustrial C_atm'
- 'albedo sensitivity'
- 'preindustrial albedo'

Benchmarking:
- When $C_{atm}=590 \ GtC$ and $\alpha=0.3$, we should get $T_{anomaly} = 0$.  
- When $C_{atm}=900 \ GtC$ and $\alpha=0.299$, we should get $T_{anomaly} \approx 1.7$.  

_Points:_ 6

In [ ]:
# Implementing the algorithm
def Diagnose_T_anomaly(C_atm, alpha, ClimateParams):
    """Returns a temperature anomaly from the carbon amount in the atmosphere"""
    """and ClimateParams' climate_sensitivity and preindust_C_atm"""

    # Extract the climate sensitivity from ClimateParams
    CS = ClimateParams['climate sensitivity']

    # Extract other information we need from ClimateParams, then calculate and return the temperature anomaly
...

# Benchmarking
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Actual temperature from temperature anomaly
Algorithm: $T(actual,C) = (T_{anomaly}+14) $. This algorithm doesn't require any climate parameters, so only T_anomaly is in the argument list. 

Benchmarking:
- return a temperature of 14 degrees C if the temperature anomaly is zero
- return a temperature of 15 degrees C if the temperature anomaly is +1

_Points:_ 2

In [ ]:
# Implementing the algorithm
def Diagnose_actual_temperature(T_anomaly):
    """Returns degrees C from a temperature anomaly"""
...

# Benchmarking
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Ocean surface pH from C_atm
Algorithm: 

$$
pH = -log_{10}(\frac {C_{atm}}{C_{atm,pre-industrial}})+pH_{pre-industrial}
$$

For this function, you'll need to extract some variables from the ClimateParams dictionary:
- "preindustrial pH"
- "preindustrial C_atm"  

Notice that we've added the concentration of carbon in the atmosphere to the argument list (as shown below). Also, numpy's the natural log is np.log; for log-base-ten (which is needed here), you'll need to use np.log10.

Benchmarking:
- When C_atm is the preindustrial value (590 GtC), the resulting pH should be $8.2$.
- When C_atm is double the preindustrial value (1280 GtC), the resulting pH should be about $7.9$. 

_Points:_ 5

In [ ]:
# Implementing the algorithm as a Python function
def Diagnose_OceanSurfacepH(C_atm,ClimateParams):
    """Returns ocean pH from the carbon amount in the atmosphere"""
    """and ClimateParams' preindust_pH and preindust_C_atm"""
...

# Benchmarking
...

<!-- END QUESTION -->

### Validating and finishing up
Assuming all this has gone smoothly, don't forget to do a Kernel/Restart & Run All, validate, and submit.